# Data Ingestion

In [11]:
import zipfile
import cv2
import numpy as np

data_set = '../data_raw/deepfake-and-real-images.zip'
train_fake = 'Dataset/Train/Fake/'
train_real = 'Dataset/Train/Real/'
test_fake = 'Dataset/Test/Fake/'
test_real = 'Dataset/Test/Real/'
validation_fake = 'Dataset/Validation/Fake/'
validation_real = 'Dataset/Validation/Real/'

train_fake_set = []
train_real_set = []
test_fake_set = []
test_real_set = []
validation_fake_set = []
validation_real_set = []

# Data Processing

In [12]:
# Creates a function to crop the faces out of images opened by cv2
def crop_faces(image, cascade_path='haarcascade_frontalface_default.xml'):
    # Load the pre-trained Haar Cascade classifier
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + cascade_path)

    # Convert image to greyscale for better detection
    img_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Detects all faces in the image and returns coordinates (x, y, w, h)
    faces = face_cascade.detectMultiScale(img_gray, 1.1, 5)

    # Crops the image to be just the face
    cropped = []
    for (x, y, w, h) in faces:
        cropped.append(image[y:y+h, x:x+w])
    
    # Returns the cropped image(s) of just a face
    return cropped

# Read the faces from the zip file then crop them and put them in their respective lists
def get_faces(dset_path, dset_list):
    # Attempt to read the file
    try:
        with zipfile.ZipFile(data_set, 'r') as archive:
            for filename in archive.namelist():
                # Check the file in a specific directory
                if filename.startswith(dset_path):
                    # Filter out directories
                    if not filename.endswith('/'):
                        # Read the image as bytes
                        img_byte = archive.read(filename)

                        # Converts the byte string into a 1D numpy array
                        img_array = np.frombuffer(img_byte, np.uint8)

                        # Read the numpy array into a color image
                        img = cv2.imdecode(img_array, 1)

                        # Crop the image to be just the face
                        faces = crop_faces(img)

                        # Append the numpy array of the image(s) into the set
                        for face in faces:
                            dset_list.append(np.array(face))
    except zipfile.BadZipFile as error:
        print(error)

Uses the above functions to crop all of the faces in the zip file and divide them based on their respective sets

In [ ]:
# Training set
get_faces(train_real, train_real_set)
get_faces(train_fake, train_fake_set)

# Testign set
get_faces(test_real, test_real_set)
get_faces(test_fake, test_fake_set)

# Validation set
get_faces(validation_real, validation_real_set)
get_faces(validation_fake, validation_fake_set)